In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 8.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

In [ ]:
targets = [
    'Mg ', 'Al ', 'Si ', 'P ', 'K FC', 'Ca FC', 'Ti FC',
    'Cr Concentration', 'Mn FC', 'Fe FC', 'Ni Concentration',
    'Cu FC', 'Zn FC', 'Rb Concentration', 'Sr Concentration',
    'Y Concentration', 'Zr Concentration'
]

features = [
    'R', 'G', 'B',
    'Lat', 'Lon', 'Elevation-1', 'Elevation-2',
    'Band 1', 'Band 2', 'Band 3', 'Band 4',
    'Band 5', 'Band 6', 'Band 7', 'Band 8'
]

In [ ]:
df = pd.read_csv("/content/Model_with_elevation_and_band_values.csv")
df = df[features + targets].dropna()

df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna().reset_index(drop=True)

df.head()

,R,G,B,Lat,Lon,Elevation-1,Elevation-2,Band 1,Band 2,Band 3,...,Cr Concentration,Mn FC,Fe FC,Ni Concentration,Cu FC,Zn FC,Rb Concentration,Sr Concentration,Y Concentration,Zr Concentration
0,19.0,14.0,15.0,18.814707,-72.521798,51,77,0.082626,0.065365,0.055144,...,0.00828,0.052525,1.827580,0.00449,0.002138,0.004831,0.00160,0.06060,0.00134,0.00506
1,7.0,5.0,11.0,18.816330,-72.522515,56,83,0.085026,0.067925,0.059044,...,0.01015,0.056804,2.269902,0.00521,0.003256,0.005386,0.00221,0.05897,0.00147,0.00578
2,24.0,17.0,16.0,18.816524,-72.522716,57,83,0.089026,0.075905,0.069465,...,0.01042,0.059697,2.365258,0.00509,0.002851,0.005482,0.00218,0.06392,0.00148,0.00637
3,18.0,13.0,14.0,18.816950,-72.523073,55,82,0.088246,0.073345,0.066365,...,0.01357,0.063371,3.869811,0.00657,0.003916,0.006908,0.00303,0.05255,0.00137,0.00816
4,15.0,12.0,14.0,18.817015,-72.523359,55,82,0.085146,0.069145,0.063084,...,0.00939,0.068640,2.607818,0.00522,0.003186,0.005685,0.00226,0.06173,0.00150,0.00714


In [ ]:
def select_features_model_based(X_train, y_train, top_k):
    selector = ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    selector.fit(X_train, y_train)

    importances = pd.Series(
        selector.feature_importances_, index=X_train.columns
    )

    return importances.sort_values(ascending=False).head(top_k).index.tolist()

In [ ]:
def objective(trial, X, y):

    model_name = trial.suggest_categorical(
        "model", ["ET", "RF", "GB", "XGB"]
    )

    top_k = trial.suggest_int("top_k", 5, len(X.columns))

    if model_name == "ET":
        model = ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 700),
            max_depth=trial.suggest_int("max_depth", 8, 20),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "RF":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 700),
            max_depth=trial.suggest_int("max_depth", 8, 20),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "GB":
        model = GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 500),
            learning_rate=trial.suggest_float("lr", 0.01, 0.1),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            random_state=42
        )

    elif model_name == "XGB":
        model = XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 600),
            max_depth=trial.suggest_int("max_depth", 3, 6),
            learning_rate=trial.suggest_float("lr", 0.01, 0.1),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample", 0.7, 1.0),
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )


    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        selected = select_features_model_based(
            X_train, y_train, top_k
        )

        X_train = X_train[selected]
        X_test = X_test[selected]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        scores.append(r2_score(y_test, preds))

    return np.mean(scores)

In [ ]:
results = []

for target in targets:
    print(f"\n🔍 Optimizing for {target}")

    X = df[features]
    y = df[target]

    sampler = optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        lambda trial: objective(trial, X, y),
        n_trials=50,
        show_progress_bar=True
    )

    results.append({
        "Target": target,
        "Best_CV_R2": study.best_value,
        "Best_Params": study.best_params
    })

[I 2026-01-22 08:14:43,452] A new study created in memory with name: no-name-73bc8ac5-558b-434f-80fc-f0216e8dd1aa



🔍 Optimizing for Mg 


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-22 08:14:50,361] Trial 0 finished with value: 0.5083426980533401 and parameters: {'model': 'RF', 'top_k': 6, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.5083426980533401.
[I 2026-01-22 08:14:59,745] Trial 1 finished with value: 0.46377185918402547 and parameters: {'model': 'ET', 'top_k': 15, 'n_estimators': 633, 'max_depth': 10}. Best is trial 0 with value: 0.5083426980533401.
[I 2026-01-22 08:15:03,892] Trial 2 finished with value: 0.41736114482788966 and parameters: {'model': 'XGB', 'top_k': 9, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 0 with value: 0.5083426980533401.
[I 2026-01-22 08:15:12,326] Trial 3 finished with value: 0.49350741253836594 and parameters: {'model': 'RF', 'top_k': 11, 'n_estimators': 318, 'max_depth': 15}. Best is trial 0 with value: 0.5083426980533401.
[I 2026-01-22 08:15:15,972] Trial 4 finished with value: 0.373434161900007

[I 2026-01-22 08:21:24,267] A new study created in memory with name: no-name-74b3e216-c698-4406-a7f6-78c172c886d5


...

[I 2026-01-22 09:55:07,452] Trial 0 finished with value: 0.6815660697726955 and parameters: {'model': 'RF', 'top_k': 6, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.6815660697726955.
[I 2026-01-22 09:55:16,609] Trial 1 finished with value: 0.7374878722829992 and parameters: {'model': 'ET', 'top_k': 15, 'n_estimators': 633, 'max_depth': 10}. Best is trial 1 with value: 0.7374878722829992.
[I 2026-01-22 09:55:20,245] Trial 2 finished with value: 0.672020736292651 and parameters: {'model': 'XGB', 'top_k': 9, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 1 with value: 0.7374878722829992.
[I 2026-01-22 09:55:28,530] Trial 3 finished with value: 0.6982333384220614 and parameters: {'model': 'RF', 'top_k': 11, 'n_estimators': 318, 'max_depth': 15}. Best is trial 1 with value: 0.7374878722829992.
[I 2026-01-22 09:55:31,969] Trial 4 finished with value: 0.6596334943053075 an

[I 2026-01-22 10:01:48,925] A new study created in memory with name: no-name-6a9a69b7-c3fd-4d18-97f0-2d369544bc27


[I 2026-01-22 10:01:48,914] Trial 49 finished with value: 0.736236529088311 and parameters: {'model': 'ET', 'top_k': 10, 'n_estimators': 595, 'max_depth': 11}. Best is trial 41 with value: 0.7402817003289162.

🔍 Optimizing for Zr Concentration


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-22 10:01:56,007] Trial 0 finished with value: 0.4100912017485294 and parameters: {'model': 'RF', 'top_k': 6, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.4100912017485294.
[I 2026-01-22 10:02:05,191] Trial 1 finished with value: 0.4960280608115184 and parameters: {'model': 'ET', 'top_k': 15, 'n_estimators': 633, 'max_depth': 10}. Best is trial 1 with value: 0.4960280608115184.
[I 2026-01-22 10:02:10,372] Trial 2 finished with value: 0.37831906821894074 and parameters: {'model': 'XGB', 'top_k': 9, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 1 with value: 0.4960280608115184.
[I 2026-01-22 10:02:17,829] Trial 3 finished with value: 0.4334939239277779 and parameters: {'model': 'RF', 'top_k': 11, 'n_estimators': 318, 'max_depth': 15}. Best is trial 1 with value: 0.4960280608115184.
[I 2026-01-22 10:02:21,727] Trial 4 finished with value: 0.33598201200400657

In [ ]:
results_df = pd.DataFrame(results)
results_df

,Target,Best_CV_R2,Best_Params
0,Mg,0.508343,"{'model': 'RF', 'top_k': 6, 'n_estimators': 36..."
1,Al,0.662198,"{'model': 'ET', 'top_k': 10, 'n_estimators': 4..."
2,Si,0.328139,"{'model': 'RF', 'top_k': 12, 'n_estimators': 6..."
3,P,0.611973,"{'model': 'ET', 'top_k': 14, 'n_estimators': 5..."
4,K FC,0.525634,"{'model': 'ET', 'top_k': 7, 'n_estimators': 62..."
5,Ca FC,0.690469,"{'model': 'GB', 'top_k': 5, 'n_estimators': 40..."
6,Ti FC,0.366443,"{'model': 'ET', 'top_k': 12, 'n_estimators': 5..."
7,Cr Concentration,0.310817,"{'model': 'ET', 'top_k': 12, 'n_estimators': 3..."
8,Mn FC,0.584346,"{'model': 'GB', 'top_k': 7, 'n_estimators': 25..."
9,Fe FC,0.483213,"{'model': 'ET', 'top_k': 10, 'n_estimators': 5..."


In [ ]:
results_df.to_csv("Part2_Elemental_Results.csv", index=False)